# 01. AquaSynex — Bitcoin Transaction & Network Data Exploration

**SIH26146 – AI-Powered Monitoring & Analysis of Bitcoin Transaction Traffic**

> **Notice**: This notebook explores the project-generated synthetic development dataset for SIH26146.
> It contains **no real personal identities, no real wallet claims, and no real criminal data**.
> All data is modeled strictly on Bitcoin UTXO mechanics and P2P network metadata as specified in the SIH26146 challenge.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

# Path setup (relative, Linux & Windows compatible)
DB_PATH = os.path.join('..', 'database', 'aquasynex.duckdb')
SAMPLE_DIR = os.path.join('..', 'data', 'sample')

# Connect to DuckDB or fallback to direct Parquet reading
if os.path.exists(DB_PATH):
    con = duckdb.connect(DB_PATH, read_only=True)
    print(f"Connected to DuckDB: {DB_PATH}")
else:
    con = duckdb.connect()
    print(f"DuckDB file not found at {DB_PATH}, loading views directly from {SAMPLE_DIR}...")
    for tbl in ['transactions', 'transaction_inputs', 'transaction_outputs', 'network_events', 'entities', 'labels', 'sih_transactions']:
        p_path = os.path.join(SAMPLE_DIR, f'{tbl}.parquet').replace('\\', '/')
        con.execute(f"CREATE VIEW {tbl} AS SELECT * FROM '{p_path}'")


## 1. Dataset Overview & Table Record Counts

Let's inspect the record counts of all canonical and relational tables loaded in DuckDB.

In [ ]:
tables = ['transactions', 'transaction_inputs', 'transaction_outputs', 'network_events', 'entities', 'labels', 'sih_transactions']
counts = {}
for t in tables:
    cnt = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    counts[t] = cnt

df_counts = pd.DataFrame(list(counts.items()), columns=['Table Name', 'Record Count'])
print(df_counts.to_string(index=False))


## 2. Temporal Analysis & Value Statistics

We examine the time span of the dataset and total volume moved in both Satoshis and Bitcoin.

In [ ]:
time_stats = con.execute("""
    SELECT 
        MIN(timestamp) AS start_time,
        MAX(timestamp) AS end_time,
        SUM(total_output_value_satoshi) AS total_sats_moved,
        AVG(total_output_value_satoshi) AS avg_sats_per_tx,
        AVG(fee_satoshi) AS avg_fee_sats,
        MIN(fee_satoshi) AS min_fee_sats,
        MAX(fee_satoshi) AS max_fee_sats
    FROM transactions
""").df()

start = time_stats['start_time'].iloc[0]
end = time_stats['end_time'].iloc[0]
tot_btc = time_stats['total_sats_moved'].iloc[0] / 1e8
avg_btc = time_stats['avg_sats_per_tx'].iloc[0] / 1e8

print(f"Start Timestamp : {start}")
print(f"End Timestamp   : {end}")
print(f"Total BTC Moved : {tot_btc:,.4f} BTC ({time_stats['total_sats_moved'].iloc[0]:,} satoshis)")
print(f"Avg BTC / Tx    : {avg_btc:,.4f} BTC")
print(f"Avg Fee (Sats)  : {time_stats['avg_fee_sats'].iloc[0]:,.2f} satoshis")


## 3. Transaction Value Distributions

Bitcoin transaction values follow heavy-tailed, power-law distributions. We inspect the log-scaled distributions of output values and transaction fees.

In [ ]:
df_tx_vals = con.execute("""
    SELECT 
        total_output_value_satoshi,
        fee_satoshi,
        input_count,
        output_count
    FROM transactions
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Output Value Distribution (Log Satoshis)
log_vals = np.log10(df_tx_vals['total_output_value_satoshi'].clip(lower=1))
axes[0].hist(log_vals, bins=40, color='#1f77b4', edgecolor='black', alpha=0.7)
axes[0].set_title('Transaction Output Value Distribution (log10 satoshis)')
axes[0].set_xlabel('log10(satoshis)')
axes[0].set_ylabel('Transaction Count')

# Fee Distribution
axes[1].hist(df_tx_vals['fee_satoshi'], bins=40, color='#ff7f0e', edgecolor='black', alpha=0.7)
axes[1].set_title('Transaction Fee Distribution (Satoshis)')
axes[1].set_xlabel('Fee (satoshis)')
axes[1].set_ylabel('Transaction Count')

plt.tight_layout()
plt.show()


## 4. Input & Output Structures (Fan-in & Fan-out)

Examining input counts (fan-in) and output counts (fan-out). Extreme fan-in patterns indicate consolidation / fund sweeping, while high fan-out indicates structuring / dispersal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_tx_vals['input_count'], bins=range(1, 25), color='#2ca02c', edgecolor='black', alpha=0.7, align='left')
axes[0].set_title('Transaction Input Count (Fan-in)')
axes[0].set_xlabel('Input Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_tx_vals['output_count'], bins=range(1, 30), color='#d62728', edgecolor='black', alpha=0.7, align='left')
axes[1].set_title('Transaction Output Count (Fan-out)')
axes[1].set_xlabel('Output Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


## 5. Network Metadata Analysis (IP, Country, ASN)

Correlating network layer metadata (peer IP, Autonomous System Number, and Country) with transaction traffic as required by the SIH26146 specification.

In [ ]:
df_country = con.execute("""
    SELECT country, COUNT(*) AS count
    FROM network_events
    GROUP BY country
    ORDER BY count DESC
    LIMIT 10
""").df()

df_asn = con.execute("""
    SELECT asn, COUNT(*) AS count
    FROM network_events
    GROUP BY asn
    ORDER BY count DESC
    LIMIT 10
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(df_country['country'], df_country['count'], color='#9467bd', edgecolor='black', alpha=0.75)
axes[0].set_title('Top 10 Origin Countries')
axes[0].set_xlabel('Country Code')
axes[0].set_ylabel('Network Observations')

axes[1].bar([str(a) for a in df_asn['asn']], df_asn['count'], color='#8c564b', edgecolor='black', alpha=0.75)
axes[1].set_title('Top 10 Autonomous System Numbers (ASNs)')
axes[1].set_xlabel('ASN')
axes[1].set_ylabel('Network Observations')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 6. Scenario Distribution & Ground-Truth Labels

Inspecting the distribution of normal, benign high volume, and suspicious scenarios in the dataset.

In [ ]:
df_scen = con.execute("""
    SELECT 
        behavior_type,
        ground_truth_label,
        COUNT(*) AS tx_count
    FROM labels
    GROUP BY behavior_type, ground_truth_label
    ORDER BY tx_count DESC
""").df()

print(df_scen.to_string(index=False))

plt.figure(figsize=(10, 4))
colors = ['#2ca02c' if lbl == 0 else '#d62728' for lbl in df_scen['ground_truth_label']]
plt.barh(df_scen['behavior_type'], df_scen['tx_count'], color=colors, edgecolor='black', alpha=0.8)
plt.gca().invert_yaxis()
plt.title('Synthetic Behavior Scenario Distribution (Green=Benign, Red=Suspicious)')
plt.xlabel('Transaction Count')
plt.tight_layout()
plt.show()


## 7. NetworkX Graph Analysis

We construct a directed transaction network from a representative sample of transaction inputs and outputs to evaluate graph topology, degree distribution, and connected components.

In [ ]:
df_graph_in = con.execute("SELECT address, txid, amount_satoshi FROM transaction_inputs LIMIT 3000").df()
df_graph_out = con.execute("SELECT txid, address, amount_satoshi FROM transaction_outputs LIMIT 3000").df()

G = nx.DiGraph()
for _, row in df_graph_in.iterrows():
    G.add_edge(row['address'], row['txid'], weight=row['amount_satoshi'])
for _, row in df_graph_out.iterrows():
    G.add_edge(row['txid'], row['address'], weight=row['amount_satoshi'])

print(f"Sample Graph Nodes : {G.number_of_nodes():,}")
print(f"Sample Graph Edges : {G.number_of_edges():,}")

components = list(nx.weakly_connected_components(G))
print(f"Weakly Connected Components : {len(components):,}")
largest_comp = max(components, key=len)
print(f"Largest Component Size       : {len(largest_comp):,} nodes ({len(largest_comp)/G.number_of_nodes()*100:.1f}% of sample)")

# Degree distribution
in_degrees = [d for n, d in G.in_degree()]
out_degrees = [d for n, d in G.out_degree()]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(in_degrees, bins=range(0, 20), color='#17becf', edgecolor='black', alpha=0.7, align='left')
axes[0].set_title('Graph Node In-Degree Distribution')
axes[0].set_xlabel('In-Degree')
axes[0].set_ylabel('Node Count')

axes[1].hist(out_degrees, bins=range(0, 20), color='#bcbd22', edgecolor='black', alpha=0.7, align='left')
axes[1].set_title('Graph Node Out-Degree Distribution')
axes[1].set_xlabel('Out-Degree')
axes[1].set_ylabel('Node Count')

plt.tight_layout()
plt.show()


## 8. Summary of Findings & Next Steps

### Findings:
1. **UTXO Integrity**: Every transaction conserves value ($\sum \text{inputs} = \sum \text{outputs} + \text{fee}$) with positive fees and no negative satoshi amounts.
2. **Network Correlation**: Network events correlate cleanly with transaction hashes, providing rich features (IP, ASN, country) for multi-layer ML analysis.
3. **Topological Richness**: The transaction graph displays a giant connected component alongside modular subgraphs, supporting PageRank, common-input clustering, and peeling chain traversal.
4. **No Target Leakage**: Labels remain in `labels.parquet`, allowing clean separation during supervised and unsupervised model evaluation.

### Next Step (Phase 2.2 / 2.3):
- Proceed to data cleaning, schema normalization pipeline (`ml/data_pipeline/`), and feature engineering (`ml/feature_engineering/`).
- Do not begin model training until features and graph embeddings are fully constructed.